In [2]:
#Source: https://www.kaggle.com/code/yugagrawal95/market-basket-analysis-apriori-in-python

import numpy as np
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules
import warnings

# suppresses pandas warnings
warnings.filterwarnings(
    "ignore",
    message=r"datetime\.datetime\.utcnow\(\) is deprecated",
    category=DeprecationWarning
)

import pandas as pd

In [3]:
df1 = pd.read_csv('commerce.csv')
df1.head()

,InvoiceNo,StockCode,Description,Quantity,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12.0,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12.0,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12.0,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48.0,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24.0,1.25,13085.0,United Kingdom


In [4]:
df1.rename(columns={df1.columns[0]: 'InvoiceNo'}, inplace=True) #fix weird character in InvoiceNo

In [5]:
df1.Country.value_counts().head(5)

Country
United Kingdom    485692
EIRE                9670
Germany             8129
France              5772
Netherlands         2769
Name: count, dtype: int64

In [6]:
#you can subset the dataframe to just a single country
df_france = df1[df1.Country == 'France']

In [7]:
#some of transaction quantity is negative which can not be possible remove that.
df1 = df1[df1.Quantity >0]

In [8]:
#convert data in format which it require converting using pivot table and Quantity sum as values. fill 0 if any nan values

basket = pd.pivot_table(data=df1,index='InvoiceNo',columns='Description',values='Quantity', aggfunc='sum',fill_value=0)
basket.head()

Description,DOORMAT UNION JACK GUNS AND ROSES,3 STRIPEY MICE FELTCRAFT,4 PURPLE FLOCK DINNER CANDLES,ANIMAL STICKERS,BLACK PIRATE TREASURE CHEST,BROWN PIRATE TREASURE CHEST,Bank Charges,CAMPHOR WOOD PORTOBELLO MUSHROOM,CHERRY BLOSSOM DECORATIVE FLASK,FAIRY CAKE CANDLES,...,dotcom email,eurobargain invc/credit,found,found again,mailout,mailout addition,temp,tk maxx mix up with pink,update,wrong invc
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
489434,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
489435,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
489436,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
489437,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
489438,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# this is slow...
# we dont need quantity/sum we need a binary: either taken or not; if user has taken that item mark as 1 else if not taken mark as 0.

def convert_into_binary(x):
    if x > 0:
        return 1
    else:
        return 0
basket_sets = basket.map(convert_into_binary)
#remove postage item as it is just a seal which almost all transaction contain.
basket_sets.drop(columns=['POSTAGE'],inplace=True)

In [10]:
# call apriori function and pass minimum support here  - this determines the number of times in total number of transaction that item was present.
frequent_itemsets = apriori(basket_sets, min_support=0.028, use_colnames=True)
# it will generate frequent itemsets using two step approach
frequent_itemsets

c:\Users\kaili\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,support,itemsets
0,0.039810,frozenset({6 RIBBONS RUSTIC CHARM})
1,0.063619,frozenset({60 TEATIME FAIRY CAKE CASES})
2,0.041476,frozenset({72 SWEETHEART FAIRY CAKE CASES})
3,0.043143,frozenset({ANTIQUE SILVER TEA GLASS ETCHED})
4,0.067286,frozenset({ASSORTED COLOUR BIRD ORNAMENT})
...,...,...
109,0.028238,"frozenset({PACK OF 60 PINK PAISLEY CAKE CASES,..."
110,0.028714,"frozenset({60 TEATIME FAIRY CAKE CASES, PACK O..."
111,0.036714,"frozenset({RED HANGING HEART T-LIGHT HOLDER, W..."
112,0.037905,"frozenset({SWEETHEART CERAMIC TRINKET BOX, STR..."


In [11]:
# we have association rules which need to put on frequent itemset. Here we are setting based on lift and has minimum lift as 1
rules_mlxtend = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
rules_mlxtend

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({PACK OF 60 PINK PAISLEY CAKE CASES}),frozenset({60 TEATIME FAIRY CAKE CASES}),0.055000,0.063619,0.028238,0.513420,8.070223,1.0,0.024739,1.924413,0.927077,0.312434,0.480361,0.478641
1,frozenset({60 TEATIME FAIRY CAKE CASES}),frozenset({PACK OF 60 PINK PAISLEY CAKE CASES}),0.063619,0.055000,0.028238,0.443862,8.070223,1.0,0.024739,1.699219,0.935610,0.312434,0.411494,0.478641
2,frozenset({60 TEATIME FAIRY CAKE CASES}),frozenset({PACK OF 72 RETRO SPOT CAKE CASES}),0.063619,0.067143,0.028714,0.451347,6.722194,1.0,0.024443,1.700269,0.909073,0.281381,0.411858,0.439503
3,frozenset({PACK OF 72 RETRO SPOT CAKE CASES}),frozenset({60 TEATIME FAIRY CAKE CASES}),0.067143,0.063619,0.028714,0.427660,6.722194,1.0,0.024443,1.636056,0.912507,0.281381,0.388774,0.439503
4,frozenset({RED HANGING HEART T-LIGHT HOLDER}),frozenset({WHITE HANGING HEART T-LIGHT HOLDER}),0.050762,0.157905,0.036714,0.723265,4.580385,1.0,0.028699,3.042961,0.823479,0.213514,0.671373,0.477887
5,frozenset({WHITE HANGING HEART T-LIGHT HOLDER}),frozenset({RED HANGING HEART T-LIGHT HOLDER}),0.157905,0.050762,0.036714,0.232509,4.580385,1.0,0.028699,1.236807,0.928253,0.213514,0.191466,0.477887
6,frozenset({SWEETHEART CERAMIC TRINKET BOX}),frozenset({STRAWBERRY CERAMIC TRINKET BOX}),0.047667,0.078095,0.037905,0.795205,10.182500,1.0,0.034182,4.501593,0.946929,0.431436,0.777856,0.640285
7,frozenset({STRAWBERRY CERAMIC TRINKET BOX}),frozenset({SWEETHEART CERAMIC TRINKET BOX}),0.078095,0.047667,0.037905,0.485366,10.182500,1.0,0.034182,1.850506,0.978184,0.431436,0.459607,0.640285
8,frozenset({WOODEN PICTURE FRAME WHITE FINISH}),frozenset({WOODEN FRAME ANTIQUE WHITE }),0.045095,0.054286,0.028810,0.638860,11.768466,1.0,0.026361,2.618688,0.958239,0.408232,0.618129,0.584781
9,frozenset({WOODEN FRAME ANTIQUE WHITE }),frozenset({WOODEN PICTURE FRAME WHITE FINISH}),0.054286,0.045095,0.028810,0.530702,11.768466,1.0,0.026361,2.034750,0.967551,0.408232,0.508539,0.584781
